In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_ROOT = Path("data")
MODEL_DIRECTORY = Path("models")
MODEL_PATH = MODEL_DIRECTORY / "cifar10_alexnet10.pth"

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 50
RANDOM_SEED = 42

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# ============================================================
# REPRODUCTIBILITY
# ============================================================

def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# CIFAR_ALEXNET MODEL ARCHITECTURE

class CifarAlexNet(nn.Module):
    def __init__(self, number_of_classes: int = 10) -> None:
        super().__init__()

        self.features = nn.Sequential(
            # Input: 32 x 32 x 3
            nn.Conv2d(
                in_channels=3,
                out_channels=8,
                kernel_size=5,
                stride=1,
                padding=2,
                bias=True,
            ),
            nn.ReLU(inplace=False),

            # Output: 16 x 16 x 8
            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),

            nn.Conv2d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=True
            ),
            nn.ReLU(inplace=False),

            # Output: 8 x 8 x 16
            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=True,
            ),
            nn.ReLU(inplace=False),

            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=True,
            ),
            nn.ReLU(inplace=False),

            nn.Conv2d(
                in_channels=32,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=True,
            ),
            nn.ReLU(inplace=False),

            # Output: 4 x 4 x 16
            nn.MaxPool2d(
                kernel_size=2,
                stride=2,
            ),
        )

        self.classifier = nn.Sequential(
            # 16 x 4 x 4 = 256 inputs
            nn.Linear(256, 64, bias=True),
            nn.ReLU(inplace=False),

            nn.Linear(64, 32, bias=True),
            nn.ReLU(inplace=False),

            # FC8: no ReLU because these are logits
            nn.Linear(32, number_of_classes, bias=True),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(inputs)
        flattened = torch.flatten(features, start_dim=1)
        logits = self.classifier(flattened)

        return logits

# ============================================================
# DATA PIPELINE
# ============================================================

def create_data_loaders() -> tuple[DataLoader, DataLoader, DataLoader]:
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5),
            ),
        ]
    )

    full_training_dataset = datasets.CIFAR10(
        root=DATASET_ROOT,
        train=True,
        download=True,
        transform=transform,
    )

    test_dataset = datasets.CIFAR10(
        root=DATASET_ROOT,
        train=False,
        download=True,
        transform=transform,
    )

    validation_size = int(0.1 * len(full_training_dataset))
    training_size = len(full_training_dataset) - validation_size

    split_generator = torch.Generator().manual_seed(RANDOM_SEED)

    training_dataset, validation_dataset = random_split(
        full_training_dataset,
        [training_size, validation_size],
        generator=split_generator,
    )

    training_loader = DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
    )

    print(f"Training images: {len(training_dataset)}")
    print(f"Validation images: {len(validation_dataset)}")
    print(f"Test images: {len(test_dataset)}")

    return training_loader, validation_loader, test_loader

# ============================================================
# TRAINING & EVALUATION LOOP
# ============================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    model.train()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

    average_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples

    return average_loss, accuracy


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_function(logits, labels)

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

    average_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples

    return average_loss, accuracy


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    set_random_seed(RANDOM_SEED)
    MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"Training device: {device}")

    training_loader, validation_loader, test_loader = (
        create_data_loaders()
    )

    model = CifarAlexNet(number_of_classes=10).to(device)

    loss_function = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    best_validation_accuracy = 0.0

    for epoch in range(1, EPOCHS + 1):
        training_loss, training_accuracy = train_one_epoch(
            model=model,
            loader=training_loader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device,
        )

        validation_loss, validation_accuracy = evaluate(
            model=model,
            loader=validation_loader,
            loss_function=loss_function,
            device=device,
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train loss={training_loss:.4f} | "
            f"train accuracy={training_accuracy * 100:.2f}% | "
            f"validation loss={validation_loss:.4f} | "
            f"validation accuracy={validation_accuracy * 100:.2f}%"
        )

        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES,
                    "architecture": {
                        "input_shape": [3, 32, 32],
                        "conv_filters": [8, 16, 32, 32, 16],
                        "fc_outputs": [64, 32, 10],
                    },
                },
                MODEL_PATH,
            )

            print(f"Saved improved model to: {MODEL_PATH}")

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    test_loss, test_accuracy = evaluate(
        model=model,
        loader=test_loader,
        loss_function=loss_function,
        device=device,
    )

    print()
    print("========================================")
    print("Training complete")
    print(f"Best validation accuracy: "
          f"{best_validation_accuracy * 100:.2f}%")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_accuracy * 100:.2f}%")
    print(f"Model: {MODEL_PATH}")
    print("========================================")


if __name__ == "__main__":
    main()